In [1]:
import pandas as pd

combined_df = pd.read_csv("dataset_preprocessed.csv")

In [2]:
combined_df.groupby("AI_flag")["resolution_time_hours"].describe()

,count,mean,std,min,25%,50%,75%,max
AI_flag,,,,,,,,
0,4561.0,68.233450,125.151362,0.500000,2.429722,16.570278,67.979722,719.918056
1,4232.0,72.173423,128.129693,0.501111,2.690139,14.062778,82.172153,719.380000


In [3]:
from scipy.stats import mannwhitneyu

ai = combined_df[combined_df["AI_flag"] == 1]["resolution_time_hours"]
human = combined_df[combined_df["AI_flag"] == 0]["resolution_time_hours"]

mannwhitneyu(ai, human)

MannwhitneyuResult(statistic=np.float64(9895847.0), pvalue=np.float64(0.039586388719000026))

In [4]:
median_ai = ai.median()
median_human = human.median()

effect = (median_human - median_ai) / median_human * 100

effect

15.132516386434894

In [5]:
h3_table = combined_df.groupby(["pr_type", "AI_flag"])["resolution_time_hours"].median().unstack()

h3_table["difference"] = h3_table[1] - h3_table[0]
h3_table["effect_%"] = (h3_table[0] - h3_table[1]) / h3_table[0] * 100

h3_table = h3_table.rename(columns={
    0: "human_median",
    1: "ai_median"
}).round(2)

h3_table

AI_flag,human_median,ai_median,difference,effect_%
pr_type,,,,
bug_fix,18.01,21.64,3.63,-20.14
documentation,18.00,6.11,-11.89,66.04
feature,18.17,7.89,-10.28,56.58
maintenance,4.77,3.04,-1.73,36.19
other,8.44,14.40,5.96,-70.69
refactor,6.98,1.53,-5.45,78.09
test,10.14,5.91,-4.24,41.78


In [6]:
from scipy.stats import mannwhitneyu

results = {}

for pr_type in combined_df["pr_type"].unique():
    subset = combined_df[combined_df["pr_type"] == pr_type]
    
    ai = subset[subset["AI_flag"] == 1]["resolution_time_hours"]
    human = subset[subset["AI_flag"] == 0]["resolution_time_hours"]
    
    if len(ai) > 10 and len(human) > 10:
        stat, p = mannwhitneyu(ai, human)
        results[pr_type] = p

results

{'feature': np.float64(0.00025041492394275517),
 'bug_fix': np.float64(2.431306539631361e-08),
 'other': np.float64(0.0985698202076867),
 'refactor': np.float64(0.2165601639959176),
 'documentation': np.float64(0.007189695931944057),
 'maintenance': np.float64(0.8006245518287685),
 'test': np.float64(0.15373792470570521)}

In [7]:
p_values = pd.Series(results, name="p_value")

final_h3 = h3_table.merge(p_values, left_index=True, right_index=True)

final_h3

,human_median,ai_median,difference,effect_%,p_value
bug_fix,18.01,21.64,3.63,-20.14,2.431307e-08
documentation,18.00,6.11,-11.89,66.04,7.189696e-03
feature,18.17,7.89,-10.28,56.58,2.504149e-04
maintenance,4.77,3.04,-1.73,36.19,8.006246e-01
other,8.44,14.40,5.96,-70.69,9.856982e-02
refactor,6.98,1.53,-5.45,78.09,2.165602e-01
test,10.14,5.91,-4.24,41.78,1.537379e-01


In [8]:
repo_analysis = combined_df.groupby(["repo", "AI_flag"])["resolution_time_hours"].median().unstack()

repo_analysis["difference"] = repo_analysis[1] - repo_analysis[0]

repo_analysis

AI_flag,0,1,difference
repo,,,
antiwork/gumroad,51.291111,89.850972,38.559861
dotnet/aspnetcore,24.944444,4.139306,-20.805139
featureform/enrichmcp,20.546667,17.491667,-3.055000
glaredb/glaredb,8.592778,17.018889,8.426111
jina-ai/node-deepresearch,9.055139,1.220278,-7.834861
liam-hq/liam,12.873333,25.493056,12.619722
microsoft/testfx,4.375694,5.228889,0.853194
mlflow/mlflow,16.386389,25.258194,8.871806
pdfme/pdfme,19.653333,8.933889,-10.719444


In [9]:
pd.set_option("display.max_rows", None)

repo_type = combined_df.groupby(
    ["repo", "pr_type", "AI_flag"]
)["resolution_time_hours"].median().unstack()

repo_type["difference"] = repo_type[1] - repo_type[0]
repo_type["effect_%"] = (repo_type[0] - repo_type[1]) / repo_type[0] * 100

repo_type_clean = repo_type.dropna(subset=[0, 1])

repo_type_clean = repo_type_clean.rename(columns={
    0: "human_median",
    1: "ai_median"
}).round(2)

repo_type_clean.reset_index()

AI_flag,repo,pr_type,human_median,ai_median,difference,effect_%
0,antiwork/gumroad,bug_fix,59.54,92.96,33.42,-56.12
1,antiwork/gumroad,documentation,6.50,36.24,29.74,-457.85
2,antiwork/gumroad,feature,44.29,148.56,104.27,-235.45
3,antiwork/gumroad,other,25.10,28.11,3.01,-12.00
4,antiwork/gumroad,refactor,6.53,37.56,31.03,-475.02
5,antiwork/gumroad,test,137.06,98.56,-38.50,28.09
6,dotnet/aspnetcore,bug_fix,28.39,13.78,-14.61,51.46
7,dotnet/aspnetcore,documentation,27.34,19.66,-7.68,28.09
8,dotnet/aspnetcore,feature,69.00,2.93,-66.07,95.76
9,dotnet/aspnetcore,maintenance,19.20,2.39,-16.81,87.56


In [10]:
combined_df["normalized_time"] = combined_df.groupby("repo")["resolution_time_hours"].transform(
    lambda x: x / x.median()
)

combined_df.groupby(["pr_type", "AI_flag"])["normalized_time"].median().unstack()

AI_flag,0,1
pr_type,,
bug_fix,1.050330,1.188876
documentation,1.212403,0.569220
feature,1.070338,0.673448
maintenance,0.362518,0.316682
other,0.679285,0.402043
refactor,0.683459,0.193324
test,0.899021,1.120727
